In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pathlib import Path
import glob
import time
import requests
import matplotlib.pyplot as plt
from google.colab import drive
import re
!pip install rdkit --break-system-packages
from rdkit import Chem
from rdkit.Chem import MolFromSmarts
drive.mount('/content/drive')

In [ ]:
SOLVANTS = {
    'cdcl3':  {'ppm': 7.26, 'Int.': 500},
    'ccl4':   None,
    'dmso-d6': {'ppm': 2.50, 'Int.': 700},
    'd2o': {'ppm': 4.75, 'Int.': 600}
}

molecules_dict = {}

def normaliser_intensites(df):
    max_int = df['Int.'].max()
    df['Int.'] = df['Int.'] / max_int * 1000
    return df

for dossier_solvant in glob.glob('/content/drive/MyDrive/IA RMN/Molecules CSV 1H/*/'):
    nom_solvant = Path(dossier_solvant).name.lower()
    pic_solvant = SOLVANTS.get(nom_solvant, None)

    for fichier in glob.glob(f'{dossier_solvant}*.csv'):
        nom = Path(fichier).stem.split('(')[0]
        try:
            df = pd.read_csv(fichier)[['ppm', 'Int.']]
            df = normaliser_intensites(df)
            molecules_dict[nom] = {
                'df': df,
                'solvant': pic_solvant
            }
            print(f"Chargé : {nom} ({len(df)} pics) — solvant : {nom_solvant}")
        except KeyError as e:
            print(f"Erreur: Colonne manquante dans le fichier {fichier}. {e}")
            print(f"Colonnes disponibles dans {fichier}: {pd.read_csv(fichier).columns.tolist()}")
            continue # Skip this file and continue with the next one

print(f"\nTotal : {len(molecules_dict)} molécules")

Chargé : Beta-Alanine (6 pics) — solvant : d2o
Chargé : D+sucrose (35 pics) — solvant : d2o
Chargé : Citric Acid (14 pics) — solvant : d2o
Chargé : Glycine (1 pics) — solvant : d2o
Chargé : L-Ascorbic Acid (13 pics) — solvant : d2o
Chargé : Sodium Glycolate (1 pics) — solvant : d2o
Chargé : Urea (1 pics) — solvant : d2o
Chargé : 1-Cyano-2-hydroxy-3-butene (27 pics) — solvant : d2o
Chargé : Acetoacetic_acid (2 pics) — solvant : d2o
Chargé : 4-Pyridoxic_acid (3 pics) — solvant : d2o
Chargé : Argininosuccinic_acid (45 pics) — solvant : d2o
Chargé : 3-Methoxytyramine (13 pics) — solvant : d2o
Chargé : Adenosine (21 pics) — solvant : d2o
Chargé : Cyclic_AMP (10 pics) — solvant : d2o
Chargé : Deoxyuridine (31 pics) — solvant : d2o
Chargé : 1_3-Diaminopropane (9 pics) — solvant : d2o
Chargé : Iodotyrosine (21 pics) — solvant : d2o
Chargé : 1-Methylhistidine (14 pics) — solvant : d2o
Chargé : Adenine (3 pics) — solvant : d2o
Chargé : Carnosine (31 pics) — solvant : d2o
Chargé : beta-Alanine (6

In [ ]:
import re

# Names that PubChem cannot resolve automatically, mapped to a name it knows.
# Add to this dictionary any name that shows up in the failure list later on.
NOMS_MANUELS = {
    'D+sucrose': 'sucrose',
    'DCM': 'dichloromethane',
    'DMSO': 'dimethyl sulfoxide',
    'THF': 'tetrahydrofuran',
    'DMF': 'dimethylformamide',
    'N_N-Dimethylformamide': 'N,N-dimethylformamide',
    'N_N-Dimethylaniline': 'N,N-dimethylaniline',
}

def nettoyer_nom_pour_pubchem(nom):
    if nom in NOMS_MANUELS:
        return NOMS_MANUELS[nom]

    n = nom

    # 1. Strip database suffixes (e.g. Benzaldehyde_HMDB0006115 -> Benzaldehyde)
    n = re.sub(r'_HMDB\d+', '', n)
    n = re.sub(r'_\d{4,}$', '', n)

    # 2. Strip stereochemical descriptors.
    #    Original names like (2R,3R,4S,5S)-... became _2R_3R_4S_5S_-... in filenames.
    #    Rather than rebuilding the parentheses (fragile), the blocks are removed:
    #    PubChem resolves the molecule fine without them, and stereochemistry is
    #    irrelevant for functional group detection anyway.
    n = re.sub(r'_(\d*[RSEZ])(_\d*[RSEZ])*_', '_', n)   # multi-descriptor blocks: _2R_3R_
    n = re.sub(r'^_?[RSEZ]_', '', n)                     # single descriptor at the start: _R_ _Z_
    n = re.sub(r'_[RSEZ]_', '_', n)                      # single descriptor in the middle
    n = re.sub(r'\b(xi|xI|ent|rel)\b', '', n, flags=re.IGNORECASE)  # non-standard stereo prefixes

    # 3. An underscore BETWEEN TWO DIGITS was originally a comma (e.g. 1_3- -> 1,3-).
    #    The lookbehind/lookahead makes sure only that case is affected,
    #    so Acetoacetic_acid still becomes "Acetoacetic acid" and not "Acetoacetic,acid".
    n = re.sub(r'(?<=\d)_(?=\d)', ',', n)

    # 4. Remaining underscores were spaces
    n = n.replace('_', ' ')

    # 5. Clean up leftovers: multiple spaces, stray dashes or commas at either end
    n = re.sub(r'\s+', ' ', n)
    n = re.sub(r'^[\s\-,]+', '', n)
    n = re.sub(r'[\s\-,]+$', '', n)
    n = n.strip()

    return n

In [ ]:
import requests
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
import pandas as pd

def get_smiles(nom_original):
    """Récupère le SMILES depuis PubChem. Gère plusieurs noms de champs et le rate limit."""
    nom = nettoyer_nom_pour_pubchem(nom_original)
    url = (f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
           f"{requests.utils.quote(nom)}/property/IsomericSMILES,CanonicalSMILES,SMILES/JSON")
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 429:
            time.sleep(5)
            response = requests.get(url, timeout=15)
        if response.status_code != 200:
            return None
        props = response.json()['PropertyTable']['Properties'][0]
        for champ in ['IsomericSMILES', 'CanonicalSMILES', 'SMILES']:
            if champ in props and props[champ]:
                return props[champ]
        return None
    except Exception:
        return None

In [ ]:
from rdkit.Chem import MolFromSmarts

FONCTIONS = {
    'aromatique':          '[c]',
    'alcool':              '[OX2H][CX4]',                  # OH group on sp³ carbon (excluding phenol/acid)
    'phenol':              '[OX2H][c]',                    # OH on aromatic (separate)
    'acide_carboxylique':  '[CX3](=O)[OX2H1]',
    'cetone':              '[#6][CX3](=O)[#6]',            # C=O between two carbons
    'aldehyde':            '[CX3H1](=O)[#6]',
    'amine':               '[NX3;H1,H2;!$(NC=O);!$(N=*)]', # NH/NH2 excluding amide and imine
    'ester':               '[CX3](=O)[OX2H0][#6]',
    'ether':               '[OD2]([CX4])[CX4]',
    'halogenure':          '[F,Cl,Br,I]',
    'nitrile':             '[NX1]#[CX2]',
    'amide':               '[CX3](=O)[NX3]',               # General: detects caffeine, urea, etc.
    'nitro':               '[$([NX3](=O)=O),$([NX3+](=O)[O-])]',
    'heterocycle_n':       '[nR]',
    'sulfoxyde':           '[SX3](=O)',
    'alcene':              '[CX3]=[CX3;!a]',
}

PATTERNS = {nom: MolFromSmarts(s) for nom, s in FONCTIONS.items()}

def detecter_fonctions(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {nom: int(mol.HasSubstructMatch(pat)) for nom, pat in PATTERNS.items()}

In [ ]:
molecules_noms = list(molecules_dict.keys())
print(f"Traitement de {len(molecules_noms)} molécules...\n")

annotations = []
smiles_dict = {}
echecs_smiles = []       # name without SMILES found
echecs_parsing = []      # SMILES found, but RDKit cannot parse it

for i, nom in enumerate(molecules_noms):
    smiles = get_smiles(nom)
    time.sleep(0.2)

    if smiles is None:
        echecs_smiles.append(nom)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(molecules_noms)}...")
        continue

    fonctions = detecter_fonctions(smiles)
    if fonctions is None:
        echecs_parsing.append((nom, smiles))
        continue

    smiles_dict[nom] = smiles
    fonctions['molecule'] = nom
    fonctions['smiles'] = smiles
    annotations.append(fonctions)

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(molecules_noms)} traitées | "
              f"{len(annotations)} OK | {len(echecs_smiles)} échecs SMILES")

df_annotations = pd.DataFrame(annotations).set_index('molecule')

print(f"\n=== Résultat ===")
print(f"Annotées avec succès : {len(df_annotations)}")
print(f"SMILES non trouvé    : {len(echecs_smiles)}")
print(f"Parsing RDKit échoué : {len(echecs_parsing)}")

Traitement de 1385 molécules...

  50/1385 traitées | 49 OK | 1 échecs SMILES
  100/1385 traitées | 97 OK | 3 échecs SMILES
  150/1385...
  200/1385 traitées | 192 OK | 8 échecs SMILES
  250/1385 traitées | 236 OK | 14 échecs SMILES
  300/1385 traitées | 286 OK | 14 échecs SMILES
  350/1385 traitées | 336 OK | 14 échecs SMILES
  400/1385 traitées | 385 OK | 15 échecs SMILES
  450/1385 traitées | 434 OK | 16 échecs SMILES
  500/1385 traitées | 480 OK | 20 échecs SMILES
  550/1385 traitées | 520 OK | 30 échecs SMILES
  600/1385 traitées | 555 OK | 45 échecs SMILES
  650/1385...
  700/1385 traitées | 639 OK | 61 échecs SMILES
  750/1385 traitées | 679 OK | 71 échecs SMILES
  800/1385 traitées | 718 OK | 82 échecs SMILES
  850/1385 traitées | 751 OK | 99 échecs SMILES
  900/1385 traitées | 792 OK | 108 échecs SMILES
  950/1385 traitées | 836 OK | 114 échecs SMILES
  1000/1385 traitées | 886 OK | 114 échecs SMILES
  1050/1385 traitées | 934 OK | 116 échecs SMILES
  1100/1385...
  1150/1385 

In [ ]:
print("=== SMILES non trouvés (à ajouter dans NOMS_MANUELS si tu veux les récupérer) ===")
for nom in echecs_smiles:
    print(f"  {nom[:50]:<52} → nettoyé : '{nettoyer_nom_pour_pubchem(nom)}'")

if echecs_parsing:
    print("\n=== SMILES trouvés mais non parsables par RDKit ===")
    for nom, smi in echecs_parsing:
        print(f"  {nom[:40]:<42} → {smi[:40]}")

=== SMILES non trouvés (à ajouter dans NOMS_MANUELS si tu veux les récupérer) ===
  Adenosine_3__5_-diphosphate                          → nettoyé : 'Adenosine 3 5 -diphosphate'
  L-Threonine                                          → nettoyé : 'L-Threonine'
  Uridine_5_-diphosphate                               → nettoyé : 'Uridine 5 -diphosphate'
  2_-Deoxyguanosine_5_-monophosphate                   → nettoyé : '2 -Deoxyguanosine 5 -monophosphate'
  5_-Methylthioadenosine                               → nettoyé : '5 -Methylthioadenosine'
  Pyridoxal_5_-phosphate                               → nettoyé : 'Pyridoxal 5 -phosphate'
  Thymidine_5_-triphosphate                            → nettoyé : 'Thymidine 5 -triphosphate'
  Pyridoxal_5_-phosphate_HMDB0001491                   → nettoyé : 'Pyridoxal 5 -phosphate'
  3_-Sialyllactose                                     → nettoyé : '3 -Sialyllactose'
  3_-Sialyllactose_HMDB0000825                         → nettoyé : '3 -Sialyllactose'
  

In [ ]:
nouveaux_ok = []
encore_echecs = []

for nom in echecs_smiles:
    smiles = get_smiles(nom)
    time.sleep(0.2)
    if smiles is None:
        encore_echecs.append(nom)
        continue
    fonctions = detecter_fonctions(smiles)
    if fonctions is None:
        encore_echecs.append(nom)
        continue
    smiles_dict[nom] = smiles
    fonctions['molecule'] = nom
    fonctions['smiles'] = smiles
    annotations.append(fonctions)
    nouveaux_ok.append(nom)

df_annotations = pd.DataFrame(annotations).set_index('molecule')

print(f"Récupérées ce tour : {len(nouveaux_ok)}")
print(f"Échecs restants    : {len(encore_echecs)}")
print(f"Total annoté       : {len(df_annotations)}")

KeyboardInterrupt: 

In [ ]:
colonnes_fonctions = list(FONCTIONS.keys())

print("=== Distribution des fonctions ===")
distribution = df_annotations[colonnes_fonctions].sum().sort_values(ascending=False)
for fonction, count in distribution.items():
    pct = count / len(df_annotations) * 100
    print(f"  {fonction:<22} {int(count):>4} molécules ({pct:.1f}%)")

sans_fonction = df_annotations[df_annotations[colonnes_fonctions].sum(axis=1) == 0]
print(f"\nMolécules sans aucune fonction détectée : {len(sans_fonction)}")
if len(sans_fonction) > 0:
    print(sans_fonction.index.tolist()[:20])

=== Distribution des fonctions ===
  aromatique              548 molécules (44.5%)
  acide_carboxylique      353 molécules (28.7%)
  alcool                  345 molécules (28.0%)
  amine                   304 molécules (24.7%)
  heterocycle_n           251 molécules (20.4%)
  amide                   206 molécules (16.7%)
  cetone                  193 molécules (15.7%)
  ester                   171 molécules (13.9%)
  alcene                  171 molécules (13.9%)
  ether                   166 molécules (13.5%)
  halogenure              155 molécules (12.6%)
  aldehyde                142 molécules (11.5%)
  phenol                  136 molécules (11.0%)
  nitrile                 101 molécules (8.2%)
  sulfoxyde                57 molécules (4.6%)
  nitro                    11 molécules (0.9%)

Molécules sans aucune fonction détectée : 4
['Betaine', 'Methane', 'Propane', 'Water']


In [ ]:
# Final list of predicted function without nitro
colonnes_fonctions = [c for c in FONCTIONS.keys() if c != 'nitro']
n_fonctions = len(colonnes_fonctions)
print(f"{n_fonctions} fonctions retenues : {colonnes_fonctions}")

if 'nitro' in df_annotations.columns:
    df_annotations = df_annotations.drop(columns=['nitro'])

15 fonctions retenues : ['aromatique', 'alcool', 'phenol', 'acide_carboxylique', 'cetone', 'aldehyde', 'amine', 'ester', 'ether', 'halogenure', 'nitrile', 'amide', 'heterocycle_n', 'sulfoxyde', 'alcene']


In [ ]:
corrections = {
    'Caffeine':             {'amide': 1},
    'N_N-Dimethylformamide':{'amide': 1},
    'Urea':                 {'amide': 1},
    'Aspirin':              {'alcool': 0},
}

for molecule, valeurs in corrections.items():
    for fonction, valeur in valeurs.items():
        df_annotations.loc[molecule, fonction] = int(valeur)

In [ ]:
import numpy as np
import json
from pathlib import Path

freq = df_annotations[colonnes_fonctions].mean().values
pos_weights = (1 - freq) / freq

print("Poids par fonction :")
for f, w in zip(colonnes_fonctions, pos_weights):
    print(f"  {f:<22} freq={df_annotations[f].mean():.3f}  poids={w:.2f}")

# Save for the prediction notebook
output_dir = Path('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/')
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / 'pos_weights.json', 'w') as f:
    json.dump({'colonnes': colonnes_fonctions,
               'poids': pos_weights.tolist()}, f, indent=2)

Poids par fonction :
  aromatique             freq=0.445  poids=1.25
  alcool                 freq=0.280  poids=2.57
  phenol                 freq=0.110  poids=8.05
  acide_carboxylique     freq=0.287  poids=2.49
  cetone                 freq=0.157  poids=5.38
  aldehyde               freq=0.115  poids=7.67
  amine                  freq=0.247  poids=3.05
  ester                  freq=0.139  poids=6.20
  ether                  freq=0.135  poids=6.42
  halogenure             freq=0.126  poids=6.94
  nitrile                freq=0.082  poids=11.19
  amide                  freq=0.167  poids=4.98
  heterocycle_n          freq=0.204  poids=3.90
  sulfoxyde              freq=0.046  poids=20.60
  alcene                 freq=0.139  poids=6.20


In [ ]:
# The annotation CSV without nitro
df_annotations.to_csv('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/annotations_fonctions.csv')

# SMILES
with open('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/smiles_dict.json', 'w') as f:
    json.dump(smiles_dict, f, indent=2)

print(f"Sauvegardé : {len(df_annotations)} molécules, {n_fonctions} fonctions")

Sauvegardé : 1231 molécules, 15 fonctions
